# Routing, fallback y presupuesto: el modelo adecuado para cada tarea

**Facsímil 6 · Construir y operar** — capítulo 5 (routing, fallback y presupuestos por tarea).

Usar el modelo más potente para todo es cómodo y **ruinoso**. La mayoría de las tareas no necesitan el
cañón: una pregunta fácil la resuelve igual de bien un modelo barato que cuesta quince veces menos. En
este cuaderno montas un **router** que manda las tareas fáciles a un modelo barato y solo escala al caro
cuando hace falta, con un **plan B** (fallback) si el barato no está seguro, y un **tope de presupuesto**
por tarea. Compararás coste y calidad frente a las dos estrategias ingenuas («caro para todo» y «barato
para todo»), probarás una **cascada de tres niveles**, verás cómo el router vale lo que vale su
clasificador y dibujarás la **frontera coste-calidad** donde viven las decisiones sensatas. La diferencia
entre un sistema viable y uno que quema dinero suele estar aquí.

### Qué vas a aprender
- Por qué «el modelo más potente para todo» (y también «el más barato para todo») son malas estrategias.
- Cómo un **router** decide a qué modelo va cada tarea, y por qué importa el **umbral de confianza**.
- Qué es un **fallback** y por qué debe dispararse por una **señal observable** (la confianza), nunca por
  «saber» si el modelo acertó (que en la realidad no lo sabes).
- Cómo un **presupuesto por tarea** evita que un caso testarudo se lleve el gasto de cien.
- Cómo una **cascada de tres niveles** abre otro punto del equilibrio coste-calidad.
- Por qué el router **vale lo que vale el clasificador**, y qué pasa cuando el caro encarece.
- A dibujar la **frontera coste-calidad** para elegir con criterio, no por corazonada.

### Cuánto cuesta
Unos 18 minutos. CPU, sin claves (se simulan los modelos).


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(1)

# Tres modelos simulados. El barato es rapido y suele acertar en lo facil; el medio
# es un termino medio; el caro acierta casi siempre pero cuesta ~15x mas que el barato.
# 'cf' = confianza que reporta el modelo (lo unico OBSERVABLE en produccion).
MODELOS = {
    "barato": {"coste": 0.2/1000, "ac_facil": 0.97, "ac_dificil": 0.55, "cf_facil": 0.90, "cf_dificil": 0.45},
    "medio":  {"coste": 0.8/1000, "ac_facil": 0.98, "ac_dificil": 0.78, "cf_facil": 0.93, "cf_dificil": 0.70},
    "caro":   {"coste": 3.0/1000, "ac_facil": 0.99, "ac_dificil": 0.92, "cf_facil": 0.95, "cf_dificil": 0.85},
}
def responde(modelo, dificil):
    m = MODELOS[modelo]
    ok = np.random.random() < (m["ac_dificil"] if dificil else m["ac_facil"])
    conf = np.clip((m["cf_dificil"] if dificil else m["cf_facil"]) + np.random.normal(0, 0.12), 0, 1)
    return ok, conf, m["coste"]

tareas = np.random.random(2000) < 0.30   # True = dificil; 30% lo son
print(f"{len(tareas)} tareas, {tareas.sum()} dificiles ({100*tareas.mean():.0f}%).")
print(f"Coste por tarea -> barato ${MODELOS['barato']['coste']*1000:.1f}/1000  "
      f"medio ${MODELOS['medio']['coste']*1000:.1f}/1000  caro ${MODELOS['caro']['coste']*1000:.1f}/1000")
def evalua(estrategia):
    aciertos = costes = 0
    for d in tareas:
        ok, c = estrategia(d); aciertos += ok; costes += c
    return aciertos/len(tareas), costes


## 1. Las dos líneas base: el caro y el barato para todo

Antes de inventar nada, conviene tener contra qué comparar. Las dos estrategias ingenuas son los
extremos: **el caro para todo** (acierta mucho, paga de más en cada tarea fácil) y **el barato para
todo** (cuesta una miseria, pero se hunde en las difíciles). La buena solución vivirá entre estos dos
extremos.


In [ ]:
def todo_caro(dificil):
    ok, conf, c = responde("caro", dificil);   return ok, c
def todo_barato(dificil):
    ok, conf, c = responde("barato", dificil); return ok, c
ac_c, co_c = evalua(todo_caro)
ac_b, co_b = evalua(todo_barato)
print(f"Caro para todo:   acierto {100*ac_c:.1f}%   coste ${co_c:.4f}")
print(f"Barato para todo: acierto {100*ac_b:.1f}%   coste ${co_b:.4f}")
print(f"\nEl barato cuesta {co_c/co_b:.0f}x menos, pero pierde {100*(ac_c-ac_b):.0f} puntos de acierto. "
      f"Ninguno de los dos extremos convence.")


## 2. El router: barato por defecto, caro cuando hace falta

Un clasificador (aquí, una pista de dificultad con algo de ruido; en real, un modelo pequeño o reglas)
decide a dónde va cada tarea. Si parece fácil, al barato; si parece difícil, al caro. Y —esto es lo
importante— si el barato responde con **poca confianza**, escalamos al caro. Fíjate en que el fallback
**no** mira si el barato acertó (eso no lo sabes en producción): mira una señal observable, la
confianza. Decidir el fallback por un oráculo de acierto sería hacer trampa.


In [ ]:
def router(dificil):
    coste = 0.0
    parece_dificil = dificil if np.random.random() < 0.85 else (not dificil)  # clasificador con ruido
    modelo = "caro" if parece_dificil else "barato"
    ok, conf, c = responde(modelo, dificil); coste += c
    if modelo == "barato" and conf < 0.7:        # fallback por BAJA CONFIANZA
        ok, conf, c = responde("caro", dificil); coste += c
    return ok, coste

ac_r, co_r = evalua(router)
print(f"Caro para todo:      acierto {100*ac_c:.1f}%   coste ${co_c:.4f}")
print(f"Router + fallback:   acierto {100*ac_r:.1f}%   coste ${co_r:.4f}")
print(f"\nAhorro: {100*(1-co_r/co_c):.0f}% del coste, con una diferencia de acierto de {100*(ac_c-ac_r):+.1f} puntos.")


**Ese es el trato del routing.** Recortas una buena parte de la factura sacrificando muy poca calidad,
porque la mayoría de las tareas no necesitaban el modelo caro. El fallback por confianza te cubre las
veces que el barato se queda corto. No es magia: es no usar un camión para llevar una carta.


## 3. El umbral de confianza: el dial que regula el equilibrio

El fallback se dispara cuando la confianza del barato cae por debajo de un umbral. Ese umbral es un
**dial**: subirlo escala más tareas al caro (más calidad, más coste); bajarlo confía más en el barato
(más ahorro, más riesgo). No hay un valor mágico; hay un equilibrio que depende de cuánto te duele cada
error. Lo recorremos para verlo.


In [ ]:
def router_umbral(dificil, umbral):
    coste = 0.0
    parece = dificil if np.random.random() < 0.85 else (not dificil)
    modelo = "caro" if parece else "barato"
    ok, conf, c = responde(modelo, dificil); coste += c
    if modelo == "barato" and conf < umbral:
        ok, conf, c = responde("caro", dificil); coste += c
    return ok, coste

print("umbral de confianza | acierto | coste   (mas umbral = escala mas = mas caro)")
for u in [0.0, 0.5, 0.7, 0.9, 1.0]:
    ac = co = 0
    for d in tareas:
        ok, c = router_umbral(d, u); ac += ok; co += c
    print(f"        {u:>4.2f}        |  {100*ac/len(tareas):>5.1f}% | ${co:.4f}")
print("\nUmbral 0 = nunca escala (el mas barato). Umbral 1 = escala siempre que use el barato.")


## 4. El tope de presupuesto: que una tarea no se desboque

Un fallback mal puesto puede encadenar reintentos carísimos. Un **presupuesto por tarea** corta: si el
coste acumulado supera un límite, se para y se devuelve lo mejor que se tenga. Es una red de seguridad
económica, igual que el tope de pasos lo era para un agente.


In [ ]:
PRESUPUESTO = 4.0/1000
def router_con_tope(dificil):
    coste = 0.0; ok = False
    for modelo in ["barato", "medio", "caro"]:   # escala mientras dude y haya presupuesto
        ok, conf, c = responde(modelo, dificil); coste += c
        if conf >= 0.7 or coste >= PRESUPUESTO:
            break
    return ok, coste
ac_t, co_t = evalua(router_con_tope)
print(f"Router con tope de presupuesto: acierto {100*ac_t:.1f}%   coste ${co_t:.4f}")
print(f"Tope por tarea: ${PRESUPUESTO*1000:.1f}/1000. Evita que una tarea testaruda se lleve el gasto de cien.")


## 5. Experimento: ¿cuándo deja de compensar el router?

El routing brilla cuando la mayoría de las tareas son fáciles. Si casi todas son difíciles, casi todo
acaba yendo al caro y el ahorro se evapora. Lo medimos variando la proporción de tareas difíciles.


In [ ]:
def router_simple(dificil):
    coste = 0.0
    parece = dificil if np.random.random() < 0.85 else (not dificil)
    modelo = "caro" if parece else "barato"
    ok, conf, c = responde(modelo, dificil); coste += c
    if modelo == "barato" and conf < 0.7:
        ok, conf, c = responde("caro", dificil); coste += c
    return ok, coste

print("% tareas dificiles | coste 'caro para todo' | coste router | ahorro")
base = np.random.random(2000)
for frac in [0.1, 0.3, 0.6, 0.9]:
    tar = base < frac
    co_caro = sum(responde("caro", d)[2] for d in tar)
    co_rout = sum(router_simple(d)[1] for d in tar)
    print(f"        {frac*100:>2.0f}%         |        ${co_caro:.4f}        |   ${co_rout:.4f}  |  {100*(1-co_rout/co_caro):.0f}%")
print("\nCuanto mas faciles son las tareas, mas ahorra el router. Si casi todas son dificiles, poco margen.")


## 6. Una cascada de tres niveles: barato, medio, caro

Saltar del barato directo al caro desaprovecha un escalón intermedio. Una **cascada** prueba primero el
barato; si duda, sube al **medio**; si el medio también duda, al caro. Cada escalón resuelve lo suyo y
solo lo verdaderamente difícil llega arriba. Lo comparamos con el router de dos niveles: no siempre gana
en todo, pero suele recortar coste a cambio de algo de acierto. Es otro punto de la frontera, no un
ganador absoluto.


In [ ]:
def cascada(dificil):
    coste = 0.0; ok = False
    for modelo in ["barato", "medio", "caro"]:
        ok, conf, c = responde(modelo, dificil); coste += c
        if conf >= 0.7:           # si esta seguro, paramos aqui
            break
    return ok, coste
ac_casc, co_casc = evalua(cascada)
print(f"Router 2 niveles (barato/caro): acierto {100*ac_r:.1f}%   coste ${co_r:.4f}")
print(f"Cascada 3 niveles:              acierto {100*ac_casc:.1f}%   coste ${co_casc:.4f}")
print("\nEl escalon intermedio recorta coste; a cambio, algo de acierto se queda por el camino. Otro punto de la frontera.")


## 7. El router vale lo que vale su clasificador

Toda la magia del routing descansa en una decisión: distinguir lo fácil de lo difícil. Si ese
clasificador acierta, el router reparte bien; si es poco mejor que lanzar una moneda, manda tareas al
sitio equivocado y el ahorro se evapora o la calidad se hunde. Variamos su acierto (de 0.5, una moneda, a
0.95, casi perfecto) y miramos qué le pasa al router.


In [ ]:
def router_calidad(dificil, calidad_clf):
    coste = 0.0
    parece = dificil if np.random.random() < calidad_clf else (not dificil)
    modelo = "caro" if parece else "barato"
    ok, conf, c = responde(modelo, dificil); coste += c
    if modelo == "barato" and conf < 0.7:
        ok, conf, c = responde("caro", dificil); coste += c
    return ok, coste

print("acierto del clasificador | acierto router | coste")
for q in [0.5, 0.7, 0.85, 0.95]:
    ac = co = 0
    for d in tareas:
        ok, c = router_calidad(d, q); ac += ok; co += c
    print(f"          {q:>4.2f}           |     {100*ac/len(tareas):>5.1f}%    | ${co:.4f}")
print("\nUn clasificador 'moneda al aire' (0.5) tira el dinero o la calidad. El routing vale lo que vale el.")


## 8. ¿Y si el caro se encarece? La economía cambia

El equilibrio depende del precio. Si el modelo caro fuera diez veces más caro todavía, ahorrar enviando
al barato pasaría de ser buena idea a ser obligatorio. Subimos el coste del caro y vemos cómo crece la
ventaja del router frente a «caro para todo».


In [ ]:
coste_caro_original = MODELOS["caro"]["coste"]
print("multiplicador del caro | coste 'caro para todo' | coste router | ahorro")
for mult in [1, 2, 5, 10]:
    MODELOS["caro"]["coste"] = coste_caro_original * mult
    co_caro = sum(responde("caro", d)[2] for d in tareas)
    co_rout = sum(router_simple(d)[1] for d in tareas)
    print(f"         x{mult:<2}          |        ${co_caro:.4f}        |   ${co_rout:.4f}  |  {100*(1-co_rout/co_caro):.0f}%")
MODELOS["caro"]["coste"] = coste_caro_original   # dejamos el precio como estaba
print("\nCuanto mas caro es el caro, mas pesa cada tarea evitada: el router gana ventaja.")


## 9. La frontera coste-calidad, dibujada

Cada estrategia es un punto en el plano coste (eje X) frente a acierto (eje Y). Dibujarlas todas juntas
deja ver el compromiso de un vistazo: arriba a la izquierda (mucho acierto, poco coste) es el paraíso;
abajo a la derecha, lo que hay que evitar. La estrategia sensata no es «la que más acierta» ni «la más
barata», sino la que mejor encaja con lo que te cuesta un error.


In [ ]:
puntos = {
    "barato para todo": (co_b, ac_b),
    "caro para todo":   (co_c, ac_c),
    "router 2 niveles": (co_r, ac_r),
    "cascada 3 niveles":(co_casc, ac_casc),
    "router + tope":    (co_t, ac_t),
}
fig, ax = plt.subplots(figsize=(6.2, 4.0))
for nombre, (x, y) in puntos.items():
    ax.scatter(x, 100*y, s=90)
    ax.annotate(nombre, (x, 100*y), xytext=(6, 4), textcoords="offset points", fontsize=9)
ax.set_xlabel("coste total (USD)"); ax.set_ylabel("acierto (%)")
ax.set_title("Frontera coste-calidad: cada estrategia, un punto")
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print("Arriba a la izquierda = mucho acierto por poco dinero. Ahi viven las buenas decisiones.")


## 10. Enrutar por tipo de tarea, no solo por dificultad

En la práctica, no siempre hace falta un clasificador de dificultad: muchas veces el **tipo** de tarea ya
te dice a qué modelo va. Traducir una frase corta o clasificar un correo es trabajo de modelo barato;
redactar un informe o razonar sobre un caso legal pide el caro. Es la misma idea que en una empresa: no
mandas al director general a fotocopiar. Enrutamos por categoría y comparamos con «caro para todo».


In [ ]:
# Cada tipo de tarea trae de serie su dificultad y su modelo recomendado.
CATALOGO = {
    "traducir":  {"dificil": False, "modelo": "barato"},
    "clasificar":{"dificil": False, "modelo": "barato"},
    "resumir":   {"dificil": False, "modelo": "medio"},
    "redactar":  {"dificil": True,  "modelo": "caro"},
    "razonar":   {"dificil": True,  "modelo": "caro"},
}
tipos = list(CATALOGO)
trabajo = np.random.choice(tipos, 2000, p=[0.30, 0.30, 0.20, 0.12, 0.08])
ac_cat = co_cat = 0
ac_tc  = co_tc  = 0
for t in trabajo:
    dif = CATALOGO[t]["dificil"]
    ok, conf, c = responde(CATALOGO[t]["modelo"], dif); ac_cat += ok; co_cat += c
    ok2, conf2, c2 = responde("caro", dif);             ac_tc  += ok2; co_tc  += c2
print(f"Enrutar por tipo de tarea: acierto {100*ac_cat/2000:.1f}%   coste ${co_cat:.4f}")
print(f"Caro para todo:            acierto {100*ac_tc/2000:.1f}%   coste ${co_tc:.4f}")
print(f"\nAhorro: {100*(1-co_cat/co_tc):.0f}%. A veces el 'clasificador' es simplemente saber que clase de tarea es.")


## 11. Pruébalo tú

1. **Empeora el clasificador** (baja `0.85` a `0.6` en `router`): si no sabes distinguir fácil de difícil,
   el routing pierde sentido. Compáralo con lo que viste en la sección 7.
2. **Cambia el umbral de confianza** del fallback (0.7) a 0.5 y a 0.9 en `router_umbral`. ¿Cómo se mueve
   el punto en la frontera coste-calidad?
3. **Sube el coste del caro** a 30× en la sección 8. ¿Cuánto más agresivo conviene ser enviando al barato?
4. **Añade un cuarto modelo** «mini» aún más barato y mételo como primer escalón de la `cascada`. ¿Mejora
   el equilibrio o solo añade ruido?
5. **Reparte distinto el trabajo** de la sección 10: sube la proporción de tareas de «razonar». ¿Hasta
   dónde aguanta el ahorro cuando casi todo es difícil?
6. **Pon un tope más agresivo** (`PRESUPUESTO = 1.0/1000`): ¿cuánta calidad sacrificas por blindar el
   gasto de las tareas testarudas?


## 12. Errores comunes

- **Mandarlo todo al modelo caro.** Cómodo y ruinoso. La mayoría de tareas no lo necesitan.
- **Mandarlo todo al barato «para ahorrar».** El otro extremo: te hundes en las tareas difíciles y el
  ahorro no compensa la calidad perdida.
- **Disparar el fallback por un oráculo de acierto.** En producción no sabes si el modelo acertó; el
  fallback debe usar una señal observable (confianza, validación, error explícito).
- **No poner presupuesto por tarea.** Los reintentos pueden descontrolarse. El tope es la red de
  seguridad económica.
- **Fijar el umbral de confianza a ojo y olvidarlo.** Es el dial que regula coste y calidad; revísalo
  cuando cambie el precio de los modelos o el coste de equivocarse.
- **Olvidar que el router vale lo que vale el clasificador.** Si no distingues fácil de difícil (o de qué
  tipo es la tarea), no hay routing que te salve.


## 13. Qué te llevas

- **Enrutar** (barato por defecto, caro cuando hace falta) recorta gran parte de la factura perdiendo muy
  poca calidad: la mayoría de tareas no necesitan el modelo más potente. Los dos extremos —caro para todo
  y barato para todo— pierden, cada uno a su manera.
- El **fallback** debe decidirse por una **señal observable** (confianza), y su **umbral** es el dial que
  regula el equilibrio coste-calidad; el **presupuesto por tarea** evita que los reintentos se
  descontrolen.
- Una **cascada de tres niveles** abre otro punto de la frontera (suele recortar coste a cambio de algo de
  acierto), y a veces el mejor «clasificador» es simplemente saber **qué tipo de tarea** tienes entre
  manos.
- La pieza crítica es esa decisión de reparto: el routing vale lo que vale el clasificador. Dibujar la
  **frontera coste-calidad** te ayuda a elegir el punto sensato en vez de la corazonada.

**Para seguir:** el capítulo 7 lleva esto a despliegues progresivos (canary, rollback); el facsímil 7
mide si la calidad que sacrificas con el modelo barato es asumible para tu caso.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*